# Intro to Distributed Alignment Search (DAS)

In [ ]:
# NOTE: Notebook metadata only: records the original tutorial author.
__author__ = "Atticus Geiger"


## Contents

1. [The hierarchical equality task](#The-hierarchical-equality-task)
    1. [An Algorithm that Solves the Equality Task](#An-Algorithm-that-Solves-the-Equality-Task)
        1. [The algorithm with no intervention](#The-algorithm-with-no-intervention)
        1. [The algorithm with an intervention](#The-algorithm-with-an-intervention)
        1. [The algorithm with an interchange intervention](#The-algorithm-with-an-interchange-intervention)
    1. [Hand Crafting an MLP to Solve Hierarchical Equality](#Hand-Crafting-an-MLP-to-Solve-Hierarchical-Equality)        
    1. [Training an MLP to Solve Hierarchical Equality](#Training-an-MLP-to-Solve-Hierarchical-Equality)
1. [Causal abstraction Analysis](#Causal-abstraction)
    1. [Basic intervention: zeroing out part of a hidden layer](#Basic-intervention:-zeroing-out-part-of-a-hidden-layer)
    1. [An interchange intervention](#An-interchange-intervention)
    1. [Alignment](#Alignment)
    1. [Evaluating an Alignment](#Evaluation)
1. [Distributed Alignment Search (DAS)](#Distributed-Alignment-Search)

## Set-up

This notebook is a hands-on introduction to __causal abstraction analysis__ [Geiger*, Lu*, Icard, and Potts (2020)](https://arxiv.org/pdf/2106.02997.pdf) using __distributed alignment search__ [Geiger*, Wu*, Potts, Icard, and Goodman (2020)](https://arxiv.org/pdf/2303.02536.pdf).

In causal abstraction analysis, we assess whether trained models conform to high-level causal models that we specify, not just in terms of their input–output behavior, but also in terms of their internal dynamics. The core technique is the __interchange intervention__, in which a causal model is provided an input and then intermediate variables are fixed to take on the values they would have for a second input.

To motivate and illustrate these concepts, we're going to focus on a hierarchical equality task, building on work by [Geiger, Carstensen, Frank, and Potts (2020)](https://arxiv.org/abs/2006.07968).

In [ ]:
# NOTE: Import-check cell: if `pyvene` is missing, this installs it from the official StanfordNLP GitHub repo.
# NOTE: This pattern keeps the notebook runnable in fresh Colab/local environments.
try:
    # This library is our indicator that the required installs
    # need to be done.
    import pyvene
except ModuleNotFoundError:
    !pip install git+https://github.com/stanfordnlp/pyvene.git


In [ ]:
# NOTE: Core imports for the demo: PyTorch + datasets + metrics + pyvene APIs.
# NOTE: `CausalModel` defines the symbolic causal graph; `IntervenableModel` wraps a neural net so we can perform activation-level counterfactual interventions.
import torch
from torch.utils.data import DataLoader
from datasets import Dataset
import random
import copy
import itertools
import numpy as np
from tqdm import tqdm, trange
from sklearn.metrics import classification_report
from transformers import get_linear_schedule_with_warmup
from pyvene import CausalModel
from pyvene.models.mlp.modelings_mlp import MLPConfig
from pyvene import create_mlp_classifier
from pyvene import (
    IntervenableModel,
    VanillaIntervention,
    RotatedSpaceIntervention,
    LowRankRotatedSpaceIntervention,
    RepresentationConfig,
    IntervenableConfig,
)


In [ ]:
# NOTE: Seed all RNGs so dataset sampling and training are reproducible across runs.
seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)
# Runtime device: prefer CUDA, fallback to CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"


## The hierarchical equality task

This section builds on results presented in [Geiger, Carstensen, Frank, and Potts (2020)](https://arxiv.org/abs/2006.07968). We will use a hierarchical equality task ([Premack 1983](https://www.cambridge.org/core/services/aop-cambridge-core/content/view/7DF6F2D22838F7546AF7279679F3571D/S0140525X00015077a.pdf/div-class-title-the-codes-of-man-and-beasts-div.pdf)) to illustrate the concepts. 

We define the hierarchical equality task as follows: The input is two pairs of objects and the output is **True** if both pairs contain the same object or if both pairs contain different objects and **False** otherwise.  For example, `AABB` and `ABCD` are both labeled **True**, while `ABCC` and `BBCD` are both labeled **False**. 

## An Algorithm that Solves the Equality Task

Let $\mathcal{A}$ be the simple tree-structured algorithm that solves this task by applying a simple equality relation three times: Compute whether the first two inputs are equal, compute whether the second two inputs are equal, then compute whether the truth-valued outputs of these first two computations are equal. 

And here's a Python implementation of $\mathcal{A}$ that supports the interventions we'll want to do:

In [ ]:
# NOTE: Build a symbolic equality task as a `pyvene.CausalModel` with variables W,X,Y,Z -> WX,YZ -> O.
# NOTE: `values`, `parents`, and `functions` define the variable domains, graph edges, and structural equations used for forward/counterfactual simulation.
def randvec(n=50, lower=-1, upper=1):
    return np.array([round(random.uniform(lower, upper), 2) for i in range(n)])
embedding_dim = 2
number_of_entities = 20
variables = ["W", "X", "Y", "Z", "WX", "YZ", "O"]
reps = [randvec(embedding_dim, lower=-1, upper=1) for _ in range(number_of_entities)]
values = {variable: reps for variable in ["W", "X", "Y", "Z"]}
values["WX"] = [True, False]
values["YZ"] = [True, False]
values["O"] = [True, False]
parents = {
    "W": [],
    "X": [],
    "Y": [],
    "Z": [],
    "WX": ["W", "X"],
    "YZ": ["Y", "Z"],
    "O": ["WX", "YZ"],
}
def FILLER():
    return reps[0]
functions = {
    "W": FILLER,
    "X": FILLER,
    "Y": FILLER,
    "Z": FILLER,
    "WX": lambda x, y: np.array_equal(x, y),
    "YZ": lambda x, y: np.array_equal(x, y),
    "O": lambda x, y: x == y,
}
pos = {
    "W": (0.2, 0),
    "X": (1, 0.1),
    "Y": (2, 0.2),
    "Z": (2.8, 0),
    "WX": (1, 2),
    "YZ": (2, 2),
    "O": (1.5, 3),
}
equiv_classes = {}
equality_model = CausalModel(variables, values, parents, functions, pos=pos)


Here's a visual depiction of the algorithm:

In [ ]:
# NOTE: Inspect the causal graph to verify parent structure and topological execution order (`timesteps`).
equality_model.print_structure()
print("Timesteps:", equality_model.timesteps)


### The algorithm with no intervention

Let's first observe the behavior of the algorithm when we provide an input of the form `BBCD` with no interventions. Here is a visual depiction:

In [ ]:
# NOTE: Run a standard forward pass through the symbolic causal model (no intervention) from observed inputs.
setting = equality_model.run_forward(
    {"W": reps[0], "X": reps[0], "Y": reps[1], "Z": reps[3]}
)
print("No intervention:\n", setting, "\n")
equality_model.print_setting(setting)


### The algorithm with an intervention

Let's now see the behavior of the algorithm when we provide the input with an intervention setting **WX** to **False**. First, a visual depiction:

And then the same computation with `compute_A`:

In [ ]:
# NOTE: Demonstrate a hard intervention (`do(WX=False)`): override node `WX` and propagate downstream effects to `O`.
print(
    "Intervention setting WX to FALSE:\n",
)
equality_model.print_setting(
    equality_model.run_forward(
        {"W": reps[0], "X": reps[0], "Y": reps[1], "Z": reps[3], "WX": False}
    )
)


Notice that, in this example, even though the left two inputs are the same, the intervention has changed the intermediate prediction for those two inputs from **True** to **False**, and thus the algorithm outputs **True**, since **WX** and **YZ** are both **False**.

### The algorithm with an interchange intervention

Finally, let's observe the behavior of the algorithm when we provide the base input `BBCD` with an intervention setting **WX** to be the value it would be for the source input `ABCC`.

In [ ]:
# NOTE: Run an interchange intervention: keep base example, but copy `WX` value from a source example.
# NOTE: This is the symbolic analogue of activation patching used later in neural space.
base = {"W": reps[0], "X": reps[0], "Y": reps[1], "Z": reps[3]}
source = {"W": reps[0], "X": reps[1], "Y": reps[2], "Z": reps[2]}
setting = equality_model.run_interchange(base, {"WX": source})
equality_model.print_setting(setting)


# Hand Crafting an MLP to Solve Hierarchical Equality

Before we train a network to solve the hierarchical equality task, first consider an analytical solution where we define a neural network to have weights that are handcrafted to solve the task by implementing the algorithm $\mathcal{A}$. The network is a two layer feedforward neural network that uses the ReLU function to compute the absolute difference between two vectors. 

In [ ]:
# NOTE: Define an MLP architecture compatible with the synthetic equality task.
config = MLPConfig(
    h_dim=embedding_dim * 4,
    activation_function="relu",
    n_layer=2,
    num_classes=2,
    pdrop=0.0,
)


In [ ]:
# NOTE: `create_mlp_classifier` builds tokenizer/model objects expected by pyvene intervention wrappers.
config, tokenizer, handcrafted = create_mlp_classifier(config)


The first layer of our handcrafted model computes:

$ReLU(W_1[\mathbf{a}, \mathbf{b}, \mathbf{c}, \mathbf{d}]) = [max(\mathbf{a}-\mathbf{b}, 0), max(\mathbf{b}-\mathbf{a}, 0), max(\mathbf{c}-\mathbf{d}, 0), max(\mathbf{d}-\mathbf{c}, 0)]$


In [ ]:
# NOTE: Manually set first-layer weights so hidden features explicitly compute pairwise difference/equality cues for W vs X and Y vs Z.
W1 = [
    [1, 0, -1, 0, 0, 0, 0, 0],
    [0, 1, 0, -1, 0, 0, 0, 0],
    [-1, 0, 1, 0, 0, 0, 0, 0],
    [0, -1, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 0, -1, 0],
    [0, 0, 0, 0, 0, 1, 0, -1],
    [0, 0, 0, 0, -1, 0, 1, 0],
    [0, 0, 0, 0, 0, -1, 0, 1],
]
handcrafted.mlp.h[0].ff1.weight = torch.nn.Parameter(torch.FloatTensor(W1))
handcrafted.mlp.h[0].ff1.bias = torch.nn.Parameter(
    torch.FloatTensor([0, 0, 0, 0, 0, 0, 0, 0])
)


The second layer of our handcrafted model computes:

$ReLU(W_2ReLU(W_1[\mathbf{a}, \mathbf{b}, \mathbf{c}, \mathbf{d}])) = [|\mathbf{a}-\mathbf{b}| - |\mathbf{c}-\mathbf{d}|, |\mathbf{c}-\mathbf{d}|-|\mathbf{a}-\mathbf{b}|, |\mathbf{a}-\mathbf{b}|, |\mathbf{c}-\mathbf{d}|,0,0,0,0]$


In [ ]:
# NOTE: Set second-layer weights to combine first-layer cues into higher-level latent features for WX and YZ logic.
W2 = [
    [1, -1, 0, 1, 0, 0, 0, 0],
    [1, -1, 0, 1, 0, 0, 0, 0],
    [1, -1, 0, 1, 0, 0, 0, 0],
    [1, -1, 0, 1, 0, 0, 0, 0],
    [-1, 1, 1, 0, 0, 0, 0, 0],
    [-1, 1, 1, 0, 0, 0, 0, 0],
    [-1, 1, 1, 0, 0, 0, 0, 0],
    [-1, 1, 1, 0, 0, 0, 0, 0],
]
handcrafted.mlp.h[1].ff1.weight = torch.nn.Parameter(
    torch.FloatTensor(W2).transpose(0, 1)
)
handcrafted.mlp.h[1].ff1.bias = torch.nn.Parameter(
    torch.FloatTensor([0, 0, 0, 0, 0, 0, 0, 0])
)


The third layer of our handcrafted model computes the logits:

$W_3 ReLU(W_2ReLU(W_1[\mathbf{a}, \mathbf{b}, \mathbf{c}, \mathbf{d}])) = [||\mathbf{a}-\mathbf{b}| - |\mathbf{c}-\mathbf{d}|| -0.999999|\mathbf{a}-\mathbf{b}|-0.999999|\mathbf{c}-\mathbf{d}|, 0]$

In [ ]:
# NOTE: Set final classifier head to map engineered hidden features to binary output label O.
W3 = [[1, 0], [1, 0], [-0.999999, 0], [-0.999999, 0], [0, 0], [0, 0], [0, 0], [0, 0]]
handcrafted.score.weight = torch.nn.Parameter(torch.FloatTensor(W3).transpose(0, 1))
handcrafted.score.bias = torch.nn.Parameter(torch.FloatTensor([0, 0.00000000000001]))


We can now use the causal model of $\mathcal{A}$ that we created to generate a labeled dataset for the hierarchical equality task and show that our handcrafted network solves the task with perfect accuracy.

In [ ]:
# NOTE: Use `generate_factual_dataset` to sample i.i.d. observational examples from the causal model (no interventions).
n_examples = 100000
examples = equality_model.generate_factual_dataset(
    n_examples, equality_model.sample_input_tree_balanced
)
X = torch.stack([example['input_ids'] for example in examples])
y = torch.stack([example['labels'] for example in examples])


In [ ]:
# NOTE: Evaluate the handcrafted network on factual data to confirm it implements the intended symbolic rule.
preds = handcrafted.forward(inputs_embeds=X)
print("Train Results")
print(classification_report(y, preds[0].argmax(1)))


# Causal abstraction

The theory of **causal abstraction** describes the conditions that must hold for the high-level tree structured algorithm to be a **simplified and faithful description** of the neural network. To perform causal abstraction analysis, we need to align high-level variables in our hypothesized algorithm $\mathcal{A}$ with sets of low-level variables in the low-level neural network $\mathcal{N}$. 

In essence: $\mathcal{A}$ is a causal abstraction of a $\mathcal{N}$ if and only if $\mathcal{A}$ and $\mathcal{N}$ provides the same output for all interchange interventions that target aligned variables.

For our handcrafted network, we align the first four neurons in the first feed-forward layer with the high-level variable 'WX' and align the other four neurons in that layer with 'YZ'. Below, we create an IntervenableConfig that allows us to taget the first four and last four neurons of the first layer for an interchange intervention. 

In [ ]:
# Causal abstraction setup: define where interventions can act in the network.
# `model_type` tells pyvene which model wrapper logic to use.
config = IntervenableConfig(
    model_type=type(handcrafted),
    representations=[
        # Rep slot 0 will represent high-level variable WX.
        RepresentationConfig(
            0,  # layer index in the MLP stack
            "block_output",  # tap activations after this block
            subspace_partition=[[0, 4], [4, 8]],  # [WX dims, YZ dims]
        ),
        # Rep slot 1 will represent high-level variable YZ.
        RepresentationConfig(
            0,
            "block_output",
            subspace_partition=[[0, 4], [4, 8]],
        ),
    ],
    # VanillaIntervention = direct activation replacement (no learned transform).
    intervention_types=VanillaIntervention,
)
# Wrap the model so we can call interchange interventions at run time.
handcrafted = IntervenableModel(config, handcrafted)


Next we create a counterfactual equality dataset that includes interchange intervention examples. We first define a function that create an id for the three possible high-level interventions, namely targetting 'WX', targetting 'YZ', and targetting them both.

In [ ]:
# Map SCM intervention specs to compact IDs used inside the dataset.
def intervention_id(intervention):
    # ID 2: both WX and YZ are intervened on.
    if "WX" in intervention and "YZ" in intervention:
        return 2
    # ID 0: only WX is intervened on.
    if "WX" in intervention:
        return 0
    # ID 1: only YZ is intervened on.
    if "YZ" in intervention:
        return 1
# Number of counterfactual examples to generate.
data_size = 2048
# Batch size used when materializing intervention examples.
batch_size = 16
# Build counterfactual data from SCM:
# - base input
# - source input(s)
# - factual label
# - counterfactual label
# - intervention_id
dataset = equality_model.generate_counterfactual_dataset(
    data_size,
    intervention_id,
    batch_size,
    device=device,
    sampler=equality_model.sample_input_tree_balanced,
)


This dataset has the following components:

* `input_ids`: a regular set of train examples
* `base_labels`: a regular set of train labels
* `source_input_ids`: sets additional training inputs sets (here, two sets) for interchange interventions
* `labels`: a list of labels if interchange interventions are performed with 'source_input_ids'
* `intervention_id`: a list of intervention sites (here, all `0` corresponding to our key for "V1")

In [ ]:
# `input_ids`: base example fed to model.
print(dataset[0]["input_ids"])
# `source_input_ids`: source example(s) supplying swapped activations.
print(dataset[0]["source_input_ids"])
# `base_labels`: model output label without interchange.
print(dataset[0]["base_labels"])
# `labels`: target output after interchange intervention.
print(dataset[0]["labels"])
# `intervention_id`: routing key (WX / YZ / both).
print(dataset[0]["intervention_id"])


To evaluate the model on this dataset, we loop through batches and peform interchange interventions based on the intervention_id. 
* When the id is 0, the first four neurons in the first layer are targetted ('WX' is targetted at the high-level)
* When the id is 1, the last four neurons in the first layer are targetted ('YZ' is targetted at the high-level)
* When the id is 2, all of the neurons in the first layer are targetted ('WX' and 'YZ' are both targetted at the high-level) 

In [ ]:
# Move wrapped model to GPU.
handcrafted.to(device)
# Ensure trainable intervention params are on GPU too.
for parameter in handcrafted.get_trainable_parameters():
    parameter.to(device)
# Collect logits for all counterfactual examples.
preds = []
for batch in DataLoader(dataset, batch_size):
    # Add "sequence" axis expected by this MLP intervention wrapper.
    batch["input_ids"] = batch["input_ids"].unsqueeze(1)
    # Add source axis so we can index multiple source streams.
    batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
    # Case: intervene on both aligned high-level variables (WX and YZ).
    if batch["intervention_id"][0] == 2:
        _, counterfactual_outputs = handcrafted(
            # Base forward pass input.
            {"inputs_embeds": batch["input_ids"]},
            [
                # Source stream for rep slot 0 (WX).
                {"inputs_embeds": batch["source_input_ids"][:, 0]},
                # Source stream for rep slot 1 (YZ).
                {"inputs_embeds": batch["source_input_ids"][:, 1]},
            ],
            {
                # Route source token/unit 0 into base token/unit 0 for both rep slots.
                "sources->base": (
                    [[[0]] * batch_size, [[0]] * batch_size],
                    [[[0]] * batch_size, [[0]] * batch_size],
                )
            },
            # Choose subspace 0 for WX rep and subspace 1 for YZ rep.
            subspaces=[[[0]] * batch_size, [[1]] * batch_size],
        )
    # Case: intervene on WX only.
    elif batch["intervention_id"][0] == 0:
        _, counterfactual_outputs = handcrafted(
            {"inputs_embeds": batch["input_ids"]},
            # Provide source only for rep slot 0.
            [{"inputs_embeds": batch["source_input_ids"][:, 0]}, None],
            {"sources->base": ([[[0]] * batch_size, None], [[[0]] * batch_size, None])},
            # Apply replacement only in WX partition.
            subspaces=[[[0]] * batch_size, None],
        )
    # Case: intervene on YZ only.
    elif batch["intervention_id"][0] == 1:
        _, counterfactual_outputs = handcrafted(
            {"inputs_embeds": batch["input_ids"]},
            # Provide source only for rep slot 1.
            [None, {"inputs_embeds": batch["source_input_ids"][:, 0]}],
            {"sources->base": ([None, [[0]] * batch_size], [None, [[0]] * batch_size])},
            # Apply replacement only in YZ partition.
            subspaces=[None, [[1]] * batch_size],
        )
    # Save counterfactual logits.
    preds.append(counterfactual_outputs[0])


In [ ]:
# Merge logits across all batches.
preds = torch.cat(preds)


Below, we can see that our handcrafted neural network is a perfect implementation of the high-level algorithm.

In [ ]:
# Compare predicted counterfactual labels vs SCM counterfactual targets.
print(
    classification_report(
        torch.tensor([x["labels"] for x in dataset]).cpu(), preds.argmax(1).cpu()
    )
)


# Training an MLP to Solve Hierarchical Equality

We've now seen how to perform causal abstraction analysis on a simple handcrafted neural networks. We turn now to training a neural network to perform the hierarchical equality task with a 4 dimensional vector embedding for each object. We define an input sampler to provide an infinite stream of new entities, rather than relying on a fixed set of vector representations.

In [ ]:
# Increase entity vector dimension for the learned-model experiment.
embedding_dim = 4
def input_sampler():
    # Sample four entity vectors (W, X, Y, Z candidates).
    A = randvec(4)
    B = randvec(4)
    C = randvec(4)
    D = randvec(4)
    # Randomly choose one structural pattern.
    x = random.randint(1, 4)
    if x == 1:
        # Neither side necessarily equal.
        return {"W": A, "X": B, "Y": C, "Z": D}
    elif x == 2:
        # Both sides equal.
        return {"W": A, "X": A, "Y": B, "Z": B}
    elif x == 3:
        # Left equal, right not forced equal.
        return {"W": A, "X": A, "Y": C, "Z": D}
    elif x == 4:
        # Right equal, left not forced equal.
        return {"W": A, "X": B, "Y": C, "Z": C}


In [ ]:
# Large factual dataset for supervised training.
n_examples = 1048576
batch_size = 1024
# Sample observational (non-intervened) data from SCM.
examples = equality_model.generate_factual_dataset(n_examples, input_sampler)
# Stack model inputs.
X = torch.stack([example['input_ids'] for example in examples])
# Stack scalar class labels.
y = torch.stack([example['labels'] for example in examples])
# X = X.unsqueeze(1)


The examples in this dataset are 8-dimensional vectors: the concatenation of 4 2-dimensional vectors. Here's the first example with its label:

In [ ]:
# Inspect one factual input and its label.
X[0], y[0]


The label for this example is determined by whether the equality value for the first two inputs matches the equality value for the second two inputs:

In [ ]:
# Check left-side equality: W == X.
left = torch.equal(X[0][:embedding_dim], X[0][embedding_dim : embedding_dim * 2])
left


In [ ]:
# Check right-side equality: Y == Z.
right = torch.equal(
    X[0][embedding_dim * 2 : embedding_dim * 3], X[0][embedding_dim * 3 :]
)
right


In [ ]:
# Label rule: O = int((W==X) == (Y==Z)).
int(left == right)


We define a three layer neural network with a ReLU activation function this task:

In [ ]:
# Define a trainable MLP for the same task.
config = MLPConfig(
    h_dim=embedding_dim * 4,
    activation_function="relu",
    n_layer=3,
    num_classes=2,
    pdrop=0.0,
)
# Build pyvene-compatible model and helper objects.
config, tokenizer, trained = create_mlp_classifier(config)
# Enable training mode.
trained.train()


In [ ]:
# HF Trainer expects dataset fields; labels are one-hot here.
train_ds = Dataset.from_dict(
    {
        # Convert scalar {0,1} label to one-hot [neg,pos].
        "labels": [
            torch.FloatTensor([0, 1]) if i == 1 else torch.FloatTensor([1, 0])
            for i in y
        ],
        # Continuous input embeddings.
        "inputs_embeds": X,
    }
)


In [ ]:
from transformers import TrainingArguments, Trainer
# Standard supervised training config.
training_args = TrainingArguments(
    output_dir="test_trainer",
    evaluation_strategy="epoch",
    learning_rate=0.001,
    num_train_epochs=3,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    report_to="none",
)
# Wrap model in HF trainer API.
trainer = Trainer(
    model=trained,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=train_ds,
    # Compute classification accuracy from logits.
    compute_metrics=lambda x: {
        "accuracy": classification_report(
            x[0].argmax(1), x[1].argmax(1), output_dict=True
        )["accuracy"]
    },
)


This neural network achieves perfect performance on its train set:

In [ ]:
# Fit the model on factual data.
_ = trainer.train()


Next we create a separate causal model with vector representations distinct from those used in training:

In [ ]:
# Build a fresh SCM with unseen entity vectors for generalization tests.
variables = ["W", "X", "Y", "Z", "WX", "YZ", "O"]
# Use more entities than before and redraw random vectors.
number_of_test_entities = 100
reps = [randvec(embedding_dim) for _ in range(number_of_test_entities)]
values = {variable: reps for variable in ["W", "X", "Y", "Z"]}
values["WX"] = [True, False]
values["YZ"] = [True, False]
values["O"] = [True, False]
# Same causal graph structure as training SCM.
parents = {
    "W": [],
    "X": [],
    "Y": [],
    "Z": [],
    "WX": ["W", "X"],
    "YZ": ["Y", "Z"],
    "O": ["WX", "YZ"],
}
# Dummy function for leaf nodes (leaf values are given directly by sampled input).
def FILLER():
    return reps[0]
# Structural equations for internal/output variables.
functions = {
    "W": FILLER,
    "X": FILLER,
    "Y": FILLER,
    "Z": FILLER,
    "WX": lambda x, y: np.array_equal(x, y),
    "YZ": lambda x, y: np.array_equal(x, y),
    "O": lambda x, y: x == y,
}
# Plot coordinates only (for visualization utilities).
pos = {
    "W": (0, 0),
    "X": (1, 0.1),
    "Y": (2, 0.2),
    "Z": (3, 0),
    "WX": (1, 2),
    "YZ": (2, 2),
    "O": (1.5, 3),
}
# Test SCM object used for factual/counterfactual eval.
test_equality_model = CausalModel(variables, values, parents, functions, pos=pos)


Our trained model generalizes perfectly this test set consisting of distinct vectors:

In [ ]:
# Sample factual test examples from unseen-entity SCM.
examples = test_equality_model.generate_factual_dataset(10000, input_sampler)
print("Test Results")
# Build eval dataset in the same format as training.
test_ds = Dataset.from_dict(
    {
        "labels": [
            torch.FloatTensor([0, 1]) if example['labels'].item() == 1 else torch.FloatTensor([1, 0])
            for example in examples
        ],
        "inputs_embeds": torch.stack([example['input_ids'] for example in examples]),
    }
)
# HF trainer returns logits as first output.
test_preds = trainer.predict(test_ds)
# Keep scalar labels for report.
y_test = [example['labels'].item() for example in examples]
print(classification_report(y_test, test_preds[0].argmax(1)))


Does it implement our high-level model of the problem, though?

# Distributed Alignment Search



We previously handcrafted the weights of a network so the two high-level variables are perfectly stored in two non-overlapping sets of neurons in the first layer of the network. However, the trained network won't have axis aligned representations of high-level concepts. Rather, the two high-level variables will be encoded in multidimensional linear subspaces of the first layer in the network.  

To learn these subspaces, we define an IntervenableConfig that allows us to target the first layer of in the network after it has been rotated by an orthogonal matrix:

In [ ]:
# DAS config: learn a rotated intervention basis instead of hard-coded partitions.
config = IntervenableConfig(
    model_type=type(trained),
    representations=[
        # Rep slot 0 (aligned to one high-level variable).
        RepresentationConfig(
            0,  # layer index
            "block_output",  # hook point
            "pos",  # intervention unit type (token position)
            1,  # max units per example
            subspace_partition=None,  # let rotation discover useful subspace
            intervention_link_key=0,  # tie reps via shared link key
        ),
        # Rep slot 1 (second high-level variable).
        RepresentationConfig(
            0,
            "block_output",
            "pos",
            1,
            subspace_partition=None,
            intervention_link_key=0,
        ),
    ],
    # Learn orthogonal rotation that defines intervention subspaces.
    intervention_types=RotatedSpaceIntervention,
)


In [ ]:
# Build intervenable wrapper around the trained network.
intervenable = IntervenableModel(config, trained, use_fast=True)
# Put intervention modules on GPU.
intervenable.set_device(device)
# Freeze base network params; train only intervention params.
intervenable.disable_model_gradients()


In [ ]:
# DAS training hyperparameters.
epochs = 10
gradient_accumulation_steps = 1
total_step = 0
target_total_step = len(dataset) * epochs
t_total = int(len(dataset) * epochs)
optimizer_params = []
for k, v in intervenable.interventions.items():
    # Each intervention has a `rotate_layer`; optimize its matrix.
    optimizer_params += [{"params": v.rotate_layer.parameters()}]
    # Both reps share link key; one rotate layer is sufficient.
    break
optimizer = torch.optim.Adam(optimizer_params, lr=0.001)
# Simple accuracy for progress display.
def compute_metrics(eval_preds, eval_labels):
    total_count = 0
    correct_count = 0
    for eval_pred, eval_label in zip(eval_preds, eval_labels):
        total_count += 1
        correct_count += eval_pred == eval_label
    accuracy = float(correct_count) / float(total_count)
    return {"accuracy": accuracy}
# Counterfactual supervision loss.
def compute_loss(outputs, labels):
    CE = torch.nn.CrossEntropyLoss()
    return CE(outputs, labels)
# Shuffle by batch block to keep pre-batched intervention structure coherent.
def batched_random_sampler(data):
    batch_indices = [_ for _ in range(int(len(data) / batch_size))]
    random.shuffle(batch_indices)
    for b_i in batch_indices:
        for i in range(b_i * batch_size, (b_i + 1) * batch_size):
            yield i


In [ ]:
# Sampler can optionally condition on desired output var/value.
def input_sampler(*args, **kwargs):
    A = randvec(4)
    B = randvec(4)
    C = randvec(4)
    D = randvec(4)
    # Unconstrained sample path.
    if kwargs.get('output_var', None) is None:
        return random.choice([
            {"W": A, "X": B, "Y": C, "Z": D},
            {"W": A, "X": A, "Y": B, "Z": B},
            {"W": A, "X": A, "Y": C, "Z": D},
            {"W": A, "X": B, "Y": C, "Z": C}
        ])
    # Constrain WX=True.
    elif kwargs['output_var'] == 'WX' and kwargs['output_var_value']:
        return random.choice([
            {"W": A, "X": A, "Y": C, "Z": D},
            {"W": A, "X": A, "Y": C, "Z": C}
        ])
    # Constrain WX=False.
    elif kwargs['output_var'] == 'WX' and not kwargs['output_var_value']:
        return random.choice([
            {"W": A, "X": B, "Y": C, "Z": D},
            {"W": A, "X": B, "Y": C, "Z": C}
        ])
    # Constrain YZ=True.
    elif kwargs['output_var'] == 'YZ' and kwargs['output_var_value']:
        return random.choice([
            {"W": A, "X": B, "Y": C, "Z": C},
            {"W": A, "X": A, "Y": C, "Z": C}
        ])
    # Else: constrain YZ=False.
    else:
        return random.choice([
            {"W": A, "X": B, "Y": C, "Z": D},
            {"W": A, "X": A, "Y": C, "Z": D}
        ])


We again generate a counterfactual dataset using our high-level causal model:

In [ ]:
# Counterfactual training set for DAS rotation learning.
n_examples = 1280000
batch_size = 6400
train_dataset = equality_model.generate_counterfactual_dataset(
    n_examples, intervention_id, batch_size, sampler=input_sampler
)


Then we train the orthgonal matrix to be such that the first four dimensions in the rotated space encode the high-level variable 'WX' and the second four dimensions encode the high-level variable 'YZ'. 

Again, we check the intervention_id for each batch of training data in order to determine whether to intervene of the first four rotated dimensions ('WX' is targetted at the high-level), the last four rotated dimensions ('YZ' is targetted at the high-level), or all of the dimensions ('WX' and 'YZ' are both targetted at the high-level). 

We can train the rotation matrix such that we get perfect interchange intervention accuracy, meaning the trained network perfectly implements the high-level algorithm on the training data.

In [ ]:
# Train mode for model internals used in forward (base params remain frozen).
intervenable.model.train()  # train enables drop-off but no grads
print("intervention trainable parameters: ", intervenable.count_parameters())
train_iterator = trange(0, int(epochs), desc="Epoch")
for epoch in train_iterator:
    # Iterate over pre-generated counterfactual examples.
    epoch_iterator = tqdm(
        DataLoader(
            train_dataset,
            batch_size=batch_size,
            sampler=batched_random_sampler(train_dataset),
        ),
        desc=f"Epoch: {epoch}",
        position=0,
        leave=True,
    )
    for batch in epoch_iterator:
        # Shape normalization for intervenable forward API.
        batch["input_ids"] = batch["input_ids"].unsqueeze(1)
        batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
        batch_size = batch["input_ids"].shape[0]
        # Move tensor fields to GPU.
        for k, v in batch.items():
            if v is not None and isinstance(v, torch.Tensor):
                batch[k] = v.to(device)
        # Both WX and YZ interventions.
        if batch["intervention_id"][0] == 2:
            _, counterfactual_outputs = intervenable(
                {"inputs_embeds": batch["input_ids"]},
                [
                    {"inputs_embeds": batch["source_input_ids"][:, 0]},
                    {"inputs_embeds": batch["source_input_ids"][:, 1]},
                ],
                {
                    "sources->base": (
                        [[[0]] * batch_size, [[0]] * batch_size],
                        [[[0]] * batch_size, [[0]] * batch_size],
                    )
                },
                # In rotated space: first half dims for one variable, second half for the other.
                subspaces=[
                    [[_ for _ in range(0, embedding_dim * 2)]] * batch_size,
                    [[_ for _ in range(embedding_dim * 2, embedding_dim * 4)]]
                    * batch_size,
                ],
            )
        # WX-only intervention.
        elif batch["intervention_id"][0] == 0:
            _, counterfactual_outputs = intervenable(
                {"inputs_embeds": batch["input_ids"]},
                [{"inputs_embeds": batch["source_input_ids"][:, 0]}, None],
                {
                    "sources->base": (
                        [[[0]] * batch_size, None],
                        [[[0]] * batch_size, None],
                    )
                },
                subspaces=[
                    [[_ for _ in range(0, embedding_dim * 2)]] * batch_size,
                    None,
                ],
            )
        # YZ-only intervention.
        elif batch["intervention_id"][0] == 1:
            _, counterfactual_outputs = intervenable(
                {"inputs_embeds": batch["input_ids"]},
                [None, {"inputs_embeds": batch["source_input_ids"][:, 0]}],
                {
                    "sources->base": (
                        [None, [[0]] * batch_size],
                        [None, [[0]] * batch_size],
                    )
                },
                subspaces=[
                    None,
                    [[_ for _ in range(embedding_dim * 2, embedding_dim * 4)]]
                    * batch_size,
                ],
            )
        # Step metrics on this batch.
        eval_metrics = compute_metrics(
            counterfactual_outputs[0].argmax(1), batch["labels"].squeeze()
        )
        # Counterfactual cross-entropy loss.
        loss = compute_loss(
            counterfactual_outputs[0], batch["labels"].squeeze().to(torch.long)
        )
        epoch_iterator.set_postfix({"loss": loss, "acc": eval_metrics["accuracy"]})
        # Optional accumulation.
        if gradient_accumulation_steps > 1:
            loss = loss / gradient_accumulation_steps
        # Backprop into rotate layer(s).
        loss.backward()
        if total_step % gradient_accumulation_steps == 0:
            optimizer.step()
            intervenable.set_zero_grad()
        total_step += 1


What's more, is it generalizes unseen test data:

In [ ]:
# Held-out counterfactual set from unseen-entity SCM.
test_dataset = test_equality_model.generate_counterfactual_dataset(
    10000, intervention_id, batch_size, device=device, sampler=input_sampler
)


In [ ]:
# Collect ground-truth and predicted counterfactual labels.
eval_labels = []
eval_preds = []
with torch.no_grad():
    epoch_iterator = tqdm(DataLoader(test_dataset, batch_size), desc=f"Test")
    for step, batch in enumerate(epoch_iterator):
        # Move tensors to GPU.
        for k, v in batch.items():
            if v is not None and isinstance(v, torch.Tensor):
                batch[k] = v.to(device)
        # Shape normalization for intervention forward.
        batch["input_ids"] = batch["input_ids"].unsqueeze(1)
        batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
        # Both-variable intervention.
        if batch["intervention_id"][0] == 2:
            _, counterfactual_outputs = intervenable(
                {"inputs_embeds": batch["input_ids"]},
                [
                    {"inputs_embeds": batch["source_input_ids"][:, 0]},
                    {"inputs_embeds": batch["source_input_ids"][:, 1]},
                ],
                {
                    "sources->base": (
                        [[[0]] * batch_size, [[0]] * batch_size],
                        [[[0]] * batch_size, [[0]] * batch_size],
                    )
                },
                subspaces=[
                    [[_ for _ in range(0, embedding_dim * 2)]] * batch_size,
                    [[_ for _ in range(embedding_dim * 2, embedding_dim * 4)]]
                    * batch_size,
                ],
            )
        # WX-only intervention.
        elif batch["intervention_id"][0] == 0:
            _, counterfactual_outputs = intervenable(
                {"inputs_embeds": batch["input_ids"]},
                [{"inputs_embeds": batch["source_input_ids"][:, 0]}, None],
                {
                    "sources->base": (
                        [[[0]] * batch_size, None],
                        [[[0]] * batch_size, None],
                    )
                },
                subspaces=[
                    [[_ for _ in range(0, embedding_dim * 2)]] * batch_size,
                    None,
                ],
            )
        # YZ-only intervention.
        elif batch["intervention_id"][0] == 1:
            _, counterfactual_outputs = intervenable(
                {"inputs_embeds": batch["input_ids"]},
                [None, {"inputs_embeds": batch["source_input_ids"][:, 0]}],
                {
                    "sources->base": (
                        [None, [[0]] * batch_size],
                        [None, [[0]] * batch_size],
                    )
                },
                subspaces=[
                    None,
                    [[_ for _ in range(embedding_dim * 2, embedding_dim * 4)]]
                    * batch_size,
                ],
            )
        # Save labels and argmax predictions.
        eval_labels += [batch["labels"]]
        eval_preds += [torch.argmax(counterfactual_outputs[0], dim=1)]
# Final counterfactual classification report.
print(classification_report(torch.cat(eval_labels).cpu(), torch.cat(eval_preds).cpu()))
